**Run evidence** · task `04a_resources` · run `621543567302069` · result **SUCCESS** · executed 2026-10-19 17:12 UTC on serverless.

Exported from the Databricks job run with `_helper/export_run_evidence.py`. Source notebook: `04_resources/04a_resources`.

# 04a — Resources (SEPARATE notebook) — warehouse & permissions via Python SDK

Per the spec, infrastructure resources (SQL warehouse, permissions) live in a
**separate** notebook from the data pipeline. This notebook uses the **Databricks
Python SDK** to:
1. Ensure a **serverless** SQL warehouse exists and is usable (reuse the given one,
   or create a small serverless warehouse if it is missing).
2. Grant the required Unity Catalog privileges on the demo schema so Genie, the
   MAS endpoint, and the App can read the gold tables, functions and metric views.

It is safe to re-run (idempotent).

In [ ]:
%run ../00_config

[00_config] Using namespace: max_kohler_demo_catalog.purchasing_demo


DemoConfig(
  catalog       = 'max_kohler_demo_catalog'
  schema        = 'purchasing_demo'
  namespace     = 'max_kohler_demo_catalog.purchasing_demo'
  warehouse_id  = 'd7a3d1cfbba73c0a'
  model         = 'databricks-claude-sonnet-4-5'
  genie_space_1 = 'SAP Purchasing — Sourcing & Supplier Risk'
  genie_space_2 = 'SAP Purchasing — Purchase Order Analytics'
  mas_endpoint  = 'sap-purchasing-mas'
  app_name      = 'sap-purchasing-app'
  job_name      = 'sap_purchasing_lakeflow_pipeline'
  lakebase      = 'sap-purchasing-ops'/'purchasing_ops' (UC catalog 'sap_purchasing_ops')
)


In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import EndpointInfoWarehouseType, CreateWarehouseRequestWarehouseType

w = WorkspaceClient()
print(f"Workspace: {w.config.host}")

Workspace: https://fe-sandbox-serverless-sandbox-gs5ou3.cloud.databricks.com


## 1) Ensure a serverless SQL warehouse
We first try the configured `warehouse_id`. If it is missing, we create a small
**serverless** warehouse named after the demo. Serverless is preferred for DBSQL.

In [ ]:
warehouse_id = CFG.warehouse_id
wh = None
try:
    wh = w.warehouses.get(id=warehouse_id)
    print(f"✓ Using existing warehouse: {wh.name} (id={wh.id}, state={wh.state})")
except Exception as e:
    print(f"Configured warehouse {warehouse_id} not found ({e}); creating a serverless one.")

if wh is None:
    created = w.warehouses.create_and_wait(
        name="sap_purchasing_serverless_wh",
        cluster_size="2X-Small",
        max_num_clusters=1,
        auto_stop_mins=10,
        enable_serverless_compute=True,
        warehouse_type=CreateWarehouseRequestWarehouseType.PRO,
    )
    warehouse_id = created.id
    print(f"✓ Created serverless warehouse id={warehouse_id}")
    print(f"  NOTE: update 00_config warehouse_id widget default to {warehouse_id} if you want this to be the demo default.")

✓ Using existing warehouse: max_demo_sql (id=d7a3d1cfbba73c0a, state=State.RUNNING)


## 2) Grant Unity Catalog privileges on the demo schema
We grant USE + SELECT + EXECUTE so consumers (Genie/MAS/App and analysts) can read
tables, run the trusted SQL functions and query metric views. Adjust the principal
to your workspace group as needed. By default we grant to `account users` (broad,
good for a demo). For a locked-down demo, replace with a specific group.

In [ ]:
PRINCIPAL = "account users"  # change to a specific group for a locked-down demo

grants = [
    (f"GRANT USE CATALOG ON CATALOG {CFG.catalog} TO `{PRINCIPAL}`"),
    (f"GRANT USE SCHEMA ON SCHEMA {CFG.namespace} TO `{PRINCIPAL}`"),
    (f"GRANT SELECT ON SCHEMA {CFG.namespace} TO `{PRINCIPAL}`"),
    (f"GRANT EXECUTE ON SCHEMA {CFG.namespace} TO `{PRINCIPAL}`"),
]
for g in grants:
    try:
        spark.sql(g)
        print("✓", g)
    except Exception as e:
        print("⚠", g, "->", e)

✓ GRANT USE CATALOG ON CATALOG max_kohler_demo_catalog TO `account users`
✓ GRANT USE SCHEMA ON SCHEMA max_kohler_demo_catalog.purchasing_demo TO `account users`
✓ GRANT SELECT ON SCHEMA max_kohler_demo_catalog.purchasing_demo TO `account users`
✓ GRANT EXECUTE ON SCHEMA max_kohler_demo_catalog.purchasing_demo TO `account users`


## 3) Summary — show current grants on the schema

In [ ]:
display(spark.sql(f"SHOW GRANTS ON SCHEMA {CFG.namespace}"))

Principal,ActionType,ObjectType,ObjectKey
guy.livni@databricks.com,READ METADATA,METASTORE,metastore_aws_us_east_2_vending_machine
rupesh.maharjan@databricks.com,READ METADATA,METASTORE,metastore_aws_us_east_2_vending_machine
account users,EXECUTE,SCHEMA,max_kohler_demo_catalog.purchasing_demo
account users,SELECT,SCHEMA,max_kohler_demo_catalog.purchasing_demo
account users,USE SCHEMA,SCHEMA,max_kohler_demo_catalog.purchasing_demo
ali.karaouzene@databricks.com,READ METADATA,METASTORE,metastore_aws_us_east_2_vending_machine


In [ ]:
dbutils.notebook.exit(f"04a_resources: SUCCESS — warehouse_id={warehouse_id}, grants applied.")

Notebook exited: 04a_resources: SUCCESS — warehouse_id=d7a3d1cfbba73c0a, grants applied.
